# A5 Annotation Sampling & AI-Assisted Silver Annotation (CPU)

Menjalankan *annotation sampling* dan *silver annotation* (weak supervision) secara
reproduktif lewat `sipature_ml`. Ikuti `docs/annotation-runbook.md` dan
`docs/taxonomy-annotation-report.md` sebelum eksekusi.

Input: `data/processed/canonical_reviews.parquet` (dari notebook `02`).
Output: `data/annotations/*` (sampling audit, assignment, template, silver labels)
dan report + figure.

Catatan: label silver adalah *AI-assisted weak supervision*, bukan gold label manusia.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"

PROJECT_DIR = Path("/content/hackathon/ml")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ANNOTATION_DIR = PROJECT_DIR / "data" / "annotations"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "annotation"

DRIVE_ANNOTATION_DIR = DRIVE_ROOT / "data" / "annotations"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "annotation"

print("Drive root:", DRIVE_ROOT)
print("Sumber canonical reviews:", DRIVE_PROCESSED_DIR / "canonical_reviews.parquet")
print("Annotation dir :", ANNOTATION_DIR)
print("Processed dir  :", PROCESSED_DIR)


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


In [ ]:
import numpy
import pandas
import pyarrow
import sklearn
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)


In [ ]:
# Salin canonical_reviews.parquet (output notebook 02) dari Drive ke lokal.
import shutil
from pathlib import Path

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

source = DRIVE_PROCESSED_DIR / "canonical_reviews.parquet"
assert source.is_file(), (
    f"canonical_reviews.parquet tidak ditemukan di Drive: {source}\n"
    "Jalankan notebook 02 terlebih dahulu dan pastikan output sudah disalin ke Drive."
)

destination = PROCESSED_DIR / "canonical_reviews.parquet"
shutil.copy2(source, destination)
print("Disalin:", source.name, "->", destination)


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


In [ ]:
from sipature_ml.config import load_config, ML_ROOT

taxonomy = load_config("taxonomy")
sampling = taxonomy["sampling"]

taxonomy_path = ML_ROOT / "configs" / "taxonomy.yaml"
schema_path = ML_ROOT / "contracts" / "annotation.schema.json"
assert taxonomy_path.is_file() and schema_path.is_file(), "taxonomy/schema tidak ditemukan"

print("Taxonomy version:", taxonomy["taxonomy_version"])
print("Status:", taxonomy["status"])
print("Jumlah aspek:", len(taxonomy["aspect_definitions"]))
print("Pilot size:", sampling["pilot_size"])
print("Main size :", sampling["main_size"])
print("Double annotation rate:", sampling["double_annotation_rate"])
print("Seed:", sampling["seed"])


In [ ]:
from sipature_ml.annotation import run_annotation_sampling

sampling_summary = run_annotation_sampling(PROCESSED_DIR, ANNOTATION_DIR, REPORT_DIR)

print("Annotation version :", sampling_summary["annotation_version"])
print("Clean text pool    :", sampling_summary["clean_text_pool"])
print("Pilot unique reviews:", sampling_summary["pilot_unique_reviews"])
print("Main unique reviews :", sampling_summary["main_unique_reviews"])
print("Main annotation tasks:", sampling_summary["main_annotation_tasks"])
print("Main double annotated:", sampling_summary["main_double_annotated_reviews"])
print("Sample overlap (harus 0):", sampling_summary["sample_overlap"])
print("Destinations in main:", sampling_summary["destinations_in_main"])
print("Assignment files:")
for name in sampling_summary["assignment_files"]:
    print("  -", name)


In [ ]:
from sipature_ml.annotation import run_silver_annotation

silver_summary = run_silver_annotation(PROCESSED_DIR, ANNOTATION_DIR, REPORT_DIR)

print("Silver version :", silver_summary["silver_version"])
print("Total records  :", silver_summary["total_records"])
print("Status counts  :", silver_summary["status_counts"])
print("Mean pass agreement:", silver_summary["mean_pass_agreement"])
print("Disagreement queue records:", silver_summary["disagreement_queue_records"])
print("Main aspect support:", silver_summary["main_aspect_support"])
print("Main polarity support:", silver_summary["main_polarity_support"])
print("Main severity support:", silver_summary["main_severity_support"])


In [ ]:
from sipature_ml.quality_figures import (
    generate_annotation_figures,
    generate_silver_figures,
)

annotation_figures = generate_annotation_figures(ANNOTATION_DIR, REPORT_DIR, FIGURE_DIR)
silver_figures = generate_silver_figures(ANNOTATION_DIR, FIGURE_DIR)

print("Annotation figures:", len(annotation_figures))
for name in annotation_figures:
    print("-", name)
print("Silver figures:", len(silver_figures))
for name in silver_figures:
    print("-", name)


In [ ]:
# Salin output annotations + report + figure ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ANNOTATION_DIR, DRIVE_ANNOTATION_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


In [ ]:
# ============================================================
# RUN SUMMARY — output path, hash, dan limitations.
# ============================================================
from sipature_ml.manifest import sha256_file
from sipature_ml.config import ML_ROOT

print("TAXONOMY VERSION:", sampling_summary["annotation_version"])
print("SILVER VERSION  :", silver_summary["silver_version"])

print("\nTAXONOMY SHA256 :", sampling_summary["taxonomy_sha256"])
print("SCHEMA SHA256    :", sampling_summary["schema_sha256"])
print("SILVER SHA256    :", silver_summary["silver_sha256"])

print("\nOUTPUT ANNOTATION DIR :", ANNOTATION_DIR)
print("OUTPUT REPORT DIR     :", REPORT_DIR)
print("OUTPUT FIGURE DIR     :", FIGURE_DIR)
print("DRIVE ANNOTATION DIR  :", DRIVE_ANNOTATION_DIR)

print("\nLIMITATIONS:")
for item in silver_summary["limitations"]:
    print("  -", item)
